In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget
%pwd

'/projectnb/batmanlab/mragoza/lung-project/notebooks/copdgene'

In [2]:
import sys, os

def add_path(p: str):
    if p not in sys.path:
        sys.path.append(p)

add_path(os.environ['LP_ROOT'])
import project

add_path(os.environ['PROJECT'] + '/' + 'param_search')
import param_search as ps

ps.set_backend('sge')
ps.set_verbose(False)

# Gather examples

In [3]:
from pathlib import Path
#data_root = Path(os.environ['LP_ROOT'] / 'data' / 'COPDGene'
data_root = Path(os.environ['PRIVATE']) / 'data' / 'COPDGene'
data_root.is_dir()

True

In [4]:
import pandas as pd
subject_file = data_root / 'sample1000_2025-07-22.csv'
subject_list = list(pd.read_csv(subject_file, sep='\t').sid)
len(subject_list)

1000

In [5]:
bad_subjects = {'LE': ['23686J', '24634V']}

In [6]:
good_subjects = {
    'LE': [sid for sid in subject_list if sid not in bad_subjects['LE']]
}
len(good_subjects['LE'])

998

# Setup experiment

In [7]:
base_dir = '2026-10-07_optimize'

template = '''\
#!/bin/bash -l
#$ -N {job_name}
#$ -P batmanlab
#$ -pe omp 4
#$ -l gpus=1
#$ -l gpu_memory=32G
#$ -l h_rt=3:00:00
set -eo pipefail

mamba activate $PROJECT/mambaforge/envs/warp

export PYTHONPATH=$LP_ROOT:$PROJECT:$PYTHOPATH

python $LP_ROOT/scripts/optimize.py {config} \\
    --set dataset.subjects=[{subject}] \\
    --set dataset.variant={variant} \\
    --set dataset.pipeline_tags.mesh_generation={mesh_tag} \\
    --set dataset.pipeline_tags.forward_simulation={sim_tag} \\
    --set optimization.pde_solver.material_type={material_type} \\
    --set optimization.pde_solver.tv_reg_weight={tv_reg_weight} \\
    --set optimization.pde_solver.newton_iters={newton_iters} \\
    --set optimization.pde_solver.cg_maxiter={cg_maxiter}

'''
name_format = 'job_{params_hash}'

grid = ps.param_grid(
    config='2026-09-15_config.yaml',
    subject=subject_list,
    variant='2026-09-15',
    sim_tag='LE',
    mesh_tag='rad',
    material_type=['LE'],
    tv_reg_weight=1e-4,
    newton_iters=100,
    cg_maxiter=1000
)

len(grid)

1000

In [19]:
jobs = ps.setup(base_dir, template, name_format, grid, overwrite=False)
jobs

OSError: /projectnb/batmanlab/mragoza/lung-project/notebooks/copdgene/2026-10-07_optimize/job_a2b8824fbd83f396/run.sh already exists

In [15]:
jobs = ps.submit(jobs)

# Monitor submitted jobs

In [8]:
jobs = ps.load(base_dir)
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.newton_iters,params.cg_maxiter,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
0,job_a2b8824fbd83f396,SUBMITTED,1,7922707,None,None,grad_norm = 5.47757608e-06\n grad_norm = 5....,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,None,None,None,None,False,None,None
1,job_7b80934bf4773487,SUBMITTED,1,7922708,None,None,[step 9] loss = 7.9088e-05 (delta = 4.0189e-02...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,None,None,None,None,False,None,None
2,job_ed97dbc405ea57eb,SUBMITTED,1,7922709,None,None,[step 6] loss = 5.3521e-05 (delta = 4.4746e-02...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,None,None,None,None,False,None,None
3,job_5794ed66c2db2422,SUBMITTED,1,7922710,None,None,Optimization converged in 1 step(s)\nStart opt...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,None,None,None,None,False,None,None
4,job_390b7420e7f78caa,SUBMITTED,1,7922711,None,None,[step 1] loss = 7.3932e-04 (delta = 0.0000e+00...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,None,None,None,None,False,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,job_432946f6c90a747e,PENDING,1,7923728,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,2026-10-07T19:01:31,status,None,None,False,None,None
996,job_2e6ad9ea1ffd9131,PENDING,1,7923729,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,2026-10-07T19:01:31,status,None,None,False,None,None
997,job_fedd2bfdee45ead7,PENDING,1,7923730,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,2026-10-07T19:01:31,status,None,None,False,None,None
998,job_d2f4eac53dce3c46,PENDING,1,7923731,None,None,None,None,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,2026-10-07T19:01:31,status,None,None,False,None,None


In [9]:
%autoreload
jobs = ps.recover(jobs)
jobs = ps.status(jobs)
jobs = ps.history(jobs) # not working for SGE backend; job_state may be stale
jobs = ps.collect(jobs)
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.newton_iters,params.cg_maxiter,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
0,job_a2b8824fbd83f396,SUBMITTED,1,7922707,None,None,grad_norm = 5.47757608e-06\n grad_norm = 5....,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,None,None,None,None,False,<NA>,<NA>
1,job_7b80934bf4773487,SUBMITTED,1,7922708,None,None,[step 9] loss = 7.9088e-05 (delta = 4.0189e-02...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,None,None,None,None,False,<NA>,<NA>
2,job_ed97dbc405ea57eb,SUBMITTED,1,7922709,None,None,[step 6] loss = 5.3521e-05 (delta = 4.4746e-02...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,None,None,None,None,False,<NA>,<NA>
3,job_5794ed66c2db2422,SUBMITTED,1,7922710,None,None,Optimization converged in 1 step(s)\nStart opt...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,None,None,None,None,False,<NA>,<NA>
4,job_390b7420e7f78caa,SUBMITTED,1,7922711,None,None,[step 1] loss = 7.3932e-04 (delta = 0.0000e+00...,,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,None,None,None,None,False,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,job_432946f6c90a747e,PENDING,1,7923728,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,2026-10-08T11:59:37,status,None,None,False,<NA>,<NA>
996,job_2e6ad9ea1ffd9131,PENDING,1,7923729,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,2026-10-08T11:59:37,status,None,None,False,<NA>,<NA>
997,job_fedd2bfdee45ead7,PENDING,1,7923730,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,2026-10-08T11:59:37,status,None,None,False,<NA>,<NA>
998,job_d2f4eac53dce3c46,PENDING,1,7923731,None,None,<NA>,<NA>,/projectnb/batmanlab/mragoza/lung-project/note...,/projectnb/batmanlab/mragoza/lung-project/note...,...,100,1000,NaN,2026-10-08T11:59:37,status,None,None,False,<NA>,<NA>


In [10]:
jobs.groupby('job_state').job_name.describe()

,count,unique,top,freq
job_state,,,,
PENDING,947,947,job_41363deb7212bc63,1
RUNNING,15,15,job_24812d76241323b8,1
SUBMITTED,38,38,job_a2b8824fbd83f396,1


In [38]:
def classify_output(stdout: str):
    if pd.isnull(stdout):
        return 'No stdout file'
    elif not str(stdout).strip():
        return 'No output message'
    elif 'Done' in stdout:
        return 'Complete'
    return 'Incomplete'

def classify_error(stderr: str):
    if pd.isnull(stderr):
        return 'No stderr file'    
    elif not str(stderr).strip():
        return 'No error message'
    elif 'Segmentation fault' in stderr:
        return 'Segmentation fault'
    elif 'CUDA out of memory' in stderr:
        return 'CUDA out of memory'
    elif 'No CUDA GPUs are available' in stderr:
        return 'CUDA not available'
    elif 'Warp CUDA error' in stderr:
        return 'Warp CUDA error'
    elif 'Warp UserWarning' in stderr:
        return 'Warp user warning'
    elif 'Newton solver failed to converge' in stderr:
        return 'Failed to converge'
    elif "meshio._exceptions.ReadError" in stderr:
        return 'Failed to load mesh'
    elif 'Non-finite PDE residual' in stderr:
        return 'Non-finite PDE residual'
    return 'Uncategorized'

jobs['error_type'] = jobs.stderr.map(classify_error)
jobs['output_type'] = jobs.stdout.map(classify_output)

jobs.groupby(['output_type', 'error_type', ])[['job_name']].count()

job_name
output_type    error_type                   
Complete       No error message          668
               Warp user warning           4
Incomplete     CUDA not available          3
               Failed to load mesh         1
               No error message            7
No stdout file No stderr file            317

In [28]:
query_jobs = jobs[jobs.error_type == 'Uncategorized']
query_jobs.groupby(['params.material_type']).job_name.describe()

,count,unique,top,freq


In [17]:
job = query_jobs.iloc[0]
print(job)
print(job.stdout)
print(job.stderr)

IndexError: single positional indexer is out-of-bounds